# Data Set

1 - age (numeric)

2 - job : type of job (categorical: "admin.","unknown","unemployed","management","housemaid","entrepreneur","student",
"blue-collar","self-employed","retired","technician","services")

3 - marital : marital status (categorical: "married","divorced","single"; note: "divorced" means divorced or widowed)

4 - education (categorical: "unknown","secondary","primary","tertiary")

5 - default: has credit in default? (binary: "yes","no")

7 - housing: has housing loan? (binary: "yes","no")

8 - loan: has personal loan? (binary: "yes","no")

# related with the last contact of the current campaign:

9 - contact: contact communication type (categorical: "unknown","telephone","cellular"

10 - month: last contact month of year (categorical: "jan", "feb", "mar", …, "nov", "dec")

11 - day_of_week (Sunday,monday...

12 - duration: last contact duration, in seconds (numeric)

# other attributes:

13 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)

14 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric, -1 means client was not previously contacted)

15 - previous: number of contacts performed before this campaign and for this client (numeric)

16 - poutcome: outcome of the previous marketing campaign (categorical: "nonexistent","failure","success")


Output variable (desired target):
17 - y - has the client subscribed a term deposit? (binary: "yes","no")

In [1]:
%pip install imbalanced-learn

import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import numpy as np

Note: you may need to restart the kernel to use updated packages.


In [2]:
data=pd.read_csv("new_train.csv")
data.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,y
0,49,blue-collar,married,basic.9y,unknown,no,no,cellular,nov,wed,227,4,999,0,nonexistent,no
1,37,entrepreneur,married,university.degree,no,no,no,telephone,nov,wed,202,2,999,1,failure,no
2,78,retired,married,basic.4y,no,no,no,cellular,jul,mon,1148,1,999,0,nonexistent,yes
3,36,admin.,married,university.degree,no,yes,no,telephone,may,mon,120,2,999,0,nonexistent,no
4,59,retired,divorced,university.degree,no,no,no,cellular,jun,tue,368,2,999,0,nonexistent,no


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32950 entries, 0 to 32949
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          32950 non-null  int64 
 1   job          32950 non-null  object
 2   marital      32950 non-null  object
 3   education    32950 non-null  object
 4   default      32950 non-null  object
 5   housing      32950 non-null  object
 6   loan         32950 non-null  object
 7   contact      32950 non-null  object
 8   month        32950 non-null  object
 9   day_of_week  32950 non-null  object
 10  duration     32950 non-null  int64 
 11  campaign     32950 non-null  int64 
 12  pdays        32950 non-null  int64 
 13  previous     32950 non-null  int64 
 14  poutcome     32950 non-null  object
 15  y            32950 non-null  object
dtypes: int64(5), object(11)
memory usage: 4.0+ MB


In [4]:
data["default"].value_counts()

default
no         26007
unknown     6940
yes            3
Name: count, dtype: int64

In [5]:
datanum = data.select_dtypes(include=np.number)
datacat = data.select_dtypes(include=object)

In [6]:
datanum

,age,duration,campaign,pdays,previous
0,49,227,4,999,0
1,37,202,2,999,1
2,78,1148,1,999,0
3,36,120,2,999,0
4,59,368,2,999,0
...,...,...,...,...,...
32945,28,192,1,999,0
32946,52,64,1,999,1
32947,54,131,4,999,0
32948,29,165,1,999,0


In [7]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

In [8]:
for i in datacat.columns:
    datacat[i]=le.fit_transform(datacat[i])

In [9]:
datacat

,job,marital,education,default,housing,loan,contact,month,day_of_week,poutcome,y
0,1,1,2,1,0,0,0,7,4,1,0
1,2,1,6,0,0,0,1,7,4,0,0
2,5,1,0,0,0,0,0,3,1,1,1
3,0,1,6,0,2,0,1,6,1,1,0
4,5,0,6,0,0,0,0,4,3,1,0
...,...,...,...,...,...,...,...,...,...,...,...
32945,7,2,3,0,2,0,0,3,3,1,0
32946,9,1,5,0,2,0,0,7,0,0,0
32947,0,1,2,0,0,2,0,3,1,1,0
32948,0,1,6,0,0,0,1,6,0,1,0


In [10]:
datafinal=pd.concat([datanum,datacat],axis=1)
datafinal

,age,duration,campaign,pdays,previous,job,marital,education,default,housing,loan,contact,month,day_of_week,poutcome,y
0,49,227,4,999,0,1,1,2,1,0,0,0,7,4,1,0
1,37,202,2,999,1,2,1,6,0,0,0,1,7,4,0,0
2,78,1148,1,999,0,5,1,0,0,0,0,0,3,1,1,1
3,36,120,2,999,0,0,1,6,0,2,0,1,6,1,1,0
4,59,368,2,999,0,5,0,6,0,0,0,0,4,3,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32945,28,192,1,999,0,7,2,3,0,2,0,0,3,3,1,0
32946,52,64,1,999,1,9,1,5,0,2,0,0,7,0,0,0
32947,54,131,4,999,0,0,1,2,0,0,2,0,3,1,1,0
32948,29,165,1,999,0,0,1,6,0,0,0,1,6,0,1,0


In [11]:
X=datafinal.drop(["pdays","y"],axis=1)
y=data["y"]

In [12]:
y.value_counts()

y
no     29238
yes     3712
Name: count, dtype: int64

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

In [14]:
from sklearn.tree import DecisionTreeClassifier
dt=DecisionTreeClassifier()
dt.fit(X_train,y_train)
ypredtra=dt.predict(X_train)
ypredtest=dt.predict(X_test)

In [15]:
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
print("Training Acc",accuracy_score(y_train,ypredtra))
print("Test Acc",accuracy_score(y_test,ypredtest))
print(classification_report(y_train,ypredtra))

Training Acc 1.0
Test Acc 0.876783004552352
              precision    recall  f1-score   support

          no       1.00      1.00      1.00     17553
         yes       1.00      1.00      1.00      2217

    accuracy                           1.00     19770
   macro avg       1.00      1.00      1.00     19770
weighted avg       1.00      1.00      1.00     19770



In [16]:
print(classification_report(y_test,ypredtest))

              precision    recall  f1-score   support

          no       0.93      0.93      0.93     11685
         yes       0.46      0.47      0.47      1495

    accuracy                           0.88     13180
   macro avg       0.70      0.70      0.70     13180
weighted avg       0.88      0.88      0.88     13180



In [17]:
from imblearn.over_sampling import SMOTE
oversample = SMOTE()
X1, y1 = oversample.fit_resample(X, y)

In [18]:
X_train_S, X_test_S, y_train_S, y_test_S = train_test_split(X1, y1, test_size=0.4, random_state=42)

In [19]:
from sklearn.tree import DecisionTreeClassifier
dt1=DecisionTreeClassifier()
dt1.fit(X_train_S,y_train_S)
ypredtra1=dt.predict(X_train_S)
ypredtest1=dt.predict(X_test_S)

In [20]:
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
print("Training Acc",accuracy_score(y_train_S,ypredtra1))
print("Test Acc",accuracy_score(y_test_S,ypredtest1))
print(classification_report(y_train_S,ypredtra1))

Training Acc 0.7830126834829699
Test Acc 0.7677739301440725
              precision    recall  f1-score   support

          no       0.70      0.98      0.82     17521
         yes       0.96      0.59      0.73     17564

    accuracy                           0.78     35085
   macro avg       0.83      0.78      0.77     35085
weighted avg       0.83      0.78      0.77     35085



In [21]:
print(classification_report(y_test_S,ypredtest1))

              precision    recall  f1-score   support

          no       0.69      0.96      0.81     11717
         yes       0.94      0.57      0.71     11674

    accuracy                           0.77     23391
   macro avg       0.81      0.77      0.76     23391
weighted avg       0.81      0.77      0.76     23391



In [22]:
from imblearn.under_sampling import RandomUnderSampler
under = RandomUnderSampler()
X2,y2=under.fit_resample(X,y)

In [23]:
X_train_S1, X_test_S1, y_train_S1, y_test_S1 = train_test_split(X2, y2, test_size=0.4, random_state=42)

In [24]:
from sklearn.tree import DecisionTreeClassifier
dt2=DecisionTreeClassifier()
dt2.fit(X_train_S1,y_train_S1)
ypredtra2=dt.predict(X_train_S1)
ypredtest2=dt.predict(X_test_S1)

In [25]:
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
print("Training Acc",accuracy_score(y_train_S1,ypredtra2))
print("Test Acc",accuracy_score(y_test_S1,ypredtest2))
print(classification_report(y_train_S1,ypredtra2))
print("Test")
print(classification_report(y_test_S1,ypredtest2))

Training Acc 0.8803322855859901
Test Acc 0.8845117845117845
              precision    recall  f1-score   support

          no       0.82      0.97      0.89      2220
         yes       0.97      0.79      0.87      2234

    accuracy                           0.88      4454
   macro avg       0.89      0.88      0.88      4454
weighted avg       0.89      0.88      0.88      4454

Test
              precision    recall  f1-score   support

          no       0.82      0.98      0.89      1492
         yes       0.97      0.79      0.87      1478

    accuracy                           0.88      2970
   macro avg       0.90      0.88      0.88      2970
weighted avg       0.90      0.88      0.88      2970

